In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * MENTAL MODEL: Designing Tables for Truth, Not Convenience
 * ─────────────────────────────────────────────────────────────────────
*/

In [ ]:
/**
 * THE FOUNDATION OF DATABASE DESIGN — Before Normalization Rules
 * ─────────────────────────────────────────────────────────────────────
 *
 * TABLE OF CONTENTS:
 * 1. THE WRONG APPROACH ................ Designing for queries, not truth
 * 2. THE AFFILIATE EXAMPLE ............. Convenience vs. Correctness
 * 3. ONE FACT PER ROW .................. The first principle
 * 4. WHAT IS A "TRUTH TABLE"? .......... Single source of truth for entities
 * 5. THE CONVENIENCE TABLE TRAP ........ Deriving from truth, not replacing it
 * 6. GOLDEN RULES ...................... Key principles to remember
 * ─────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * SAMPLE DOMAIN — Affiliate System (for illustrating the mental model)
 * ─────────────────────────────────────────────────────────────────────
 *
 * BUSINESS RULES:
 *   → Each user has an affiliate code
 *   → When someone purchases using that code, the affiliate earns 15%
 *   → We need to track earnings and answer: "How much has this affiliate earned?"
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 1. THE WRONG APPROACH — Designing for queries, not truth
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ HOW MOST ENGINEERS START:                                          │
 * │                                                                    │
 * │   They ask questions first:                                        │
 * │     → How much revenue did we make?                                │
 * │     → How many students do we have?                                │
 * │     → Which course generates the most revenue?                     │
 * │     → Who is the instructor of this course?                        │
 * │                                                                    │
 * │   Then they design tables to answer those queries EASILY.          │
 * │                                                                    │
 * │   This is the WRONG APPROACH.                                      │
 * │                                                                    │
 * │ WHY IT'S WRONG:                                                    │
 * │   → You optimize for CONVENIENCE, not CORRECTNESS                  │
 * │   → You store DERIVED values as if they were facts                 │
 * │   → You make assumptions that may change                           │
 * │   → You create data that can become CORRUPTED                      │
 * │                                                                    │
 * │ THE CORRECT MENTAL MODEL:                                          │
 * │   You should NEVER design tables for convenience.                  │
 * │   You should design tables for TRUTH.                              │
 * │                                                                    │
 * │   Queries can be optimized later.                                  │
 * │   Data integrity cannot be easily fixed later.                     │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 2. THE AFFILIATE EXAMPLE — Convenience vs. Correctness
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ SCENARIO: Affiliate System                                         │
 * │   → User gets an affiliate code                                    │
 * │   → When someone purchases using that code, affiliate earns 15%    │
 * │   → Need to track earnings                                         │
 * │                                                                    │
 * │ THE CONVENIENCE DESIGN (WRONG):                                    │
 * │                                                                    │
 * │   CREATE TABLE users (                                             │
 * │     user_id SERIAL PRIMARY KEY,                                    │
 * │     affiliate_code VARCHAR(50) UNIQUE,                             │
 * │     total_affiliate_earnings DECIMAL(10,2) DEFAULT 0  ← PROBLEM!   │
 * │   );                                                               │
 * │                                                                    │
 * │   -- Every purchase:                                               │
 * │   UPDATE users                                                     │
 * │   SET total_affiliate_earnings = total_affiliate_earnings + 15%    │
 * │   WHERE affiliate_code = 'RAJ123';                                 │
 * │                                                                    │
 * │   -- Checking earnings:                                            │
 * │   SELECT total_affiliate_earnings FROM users WHERE user_id = 1;    │
 * │                                                                    │
 * │   Easy. Clean. Fast.                                               │
 * │                                                                    │
 * │   BUT WRONG.                                                       │
 * │                                                                    │
 * │ WHY THIS DESIGN IS WRONG:                                          │
 * │                                                                    │
 * │   You stored a DERIVED value as TRUTH.                             │
 * │                                                                    │
 * │   What happens if:                                                 │
 * │     → A refund happens?                                            │
 * │     → A calculation bug happens?                                   │
 * │     → Commission rules change from 15% to 20%?                     │
 * │     → A payment fails and is retried?                              │
 * │     → Someone needs to audit earnings?                             │
 * │                                                                    │
 * │   The total_affiliate_earnings column becomes CORRUPTED.           │
 * │   You cannot verify it. You cannot recalculate it.                 │
 * │   You lose the ability to audit.                                   │
 * │                                                                    │
 * │   You optimized for CONVENIENCE, not CORRECTNESS.                  │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 2.1 THE CORRECT DESIGN — One fact per row
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE CORRECT DESIGN (Storing Facts):                                │
 * │                                                                    │
 * │   CREATE TABLE affiliate_sales (                                   │
 * │     sale_id SERIAL PRIMARY KEY,                                    │
 * │     affiliate_code VARCHAR(50) NOT NULL REFERENCES users(code),    │
 * │     customer_email VARCHAR(255) NOT NULL,                          │
 * │     purchase_date TIMESTAMP NOT NULL,                              │
 * │     purchase_amount DECIMAL(10,2) NOT NULL,                        │
 * │     commission_share DECIMAL(5,2) NOT NULL,  -- e.g., 0.15         │
 * │     commission_earned DECIMAL(10,2) NOT NULL,  -- computed         │
 * │     status VARCHAR(20) NOT NULL DEFAULT 'PENDING'                  │
 * │   );                                                               │
 * │                                                                    │
 * │   -- Every purchase: store ONE ROW                                 │
 * │   INSERT INTO affiliate_sales                                      │
 * │   (affiliate_code, customer_email, purchase_date,                  │
 * │    purchase_amount, commission_share, commission_earned, status)   │
 * │   VALUES ('RAJ123', 'customer@example.com', NOW(),                 │
 * │           100.00, 0.15, 15.00, 'PAID');                            │ 
 * │                                                                    │
 * │   -- To get total earnings: COMPUTE from facts                     │
 * │   SELECT SUM(commission_earned) AS total_earnings                  │
 * │   FROM affiliate_sales                                             │
 * │   WHERE affiliate_code = 'RAJ123'                                  │
 * │     AND status = 'PAID';                                           │
 * │                                                                    │
 * │ WHY THIS DESIGN IS CORRECT:                                        │
 * │                                                                    │
 * │   → Each row = ONE FACT (one sale)                                 │
 * │   → Earnings are COMPUTED, not STORED as an accumulating column    │
 * │   → You can audit: every sale has a traceable record               │
 * │   → Refunds: just add a refund row with negative commission        │
 * │   → Rule changes: store the commission_share at time of sale       │
 * │   → Always verifiable: SUM(commission_earned) = truth              │
 * │                                                                    │
 * │   If three purchases happen, you store THREE rows.                 │
 * │   You compute truth from facts.                                    │
 * │   You do NOT store pre-aggregated assumptions.                     │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 3. ONE FACT PER ROW — The first principle
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THE FIRST PRINCIPLE:                                               │
 * │   ONE ROW SHOULD REPRESENT ONE FACT.                               │
 * │                                                                    │
 * │ EXAMPLES:                                                          │
 * │                                                                    │
 * │   ┌─────────────────────────────────────────────────────────┐      │
 * │   │ Entity        │ One row = one...                        │      │
 * │   ├───────────────┼─────────────────────────────────────────┤      │
 * │   │ Student       │ One student                             │      │
 * │   │ Course        │ One course                              │      │
 * │   │ Payment       │ One payment transaction                 │      │
 * │   │ Enrollment    │ One student enrolling in one course     │      │
 * │   │ Affiliate Sale│ One sale made through an affiliate      │      │
 * │   │ Refund        │ One refund transaction                  │      │
 * │   └───────────────┴─────────────────────────────────────────┘      │
 * │                                                                    │
 * │ WHAT THIS PREVENTS:                                                │
 * │                                                                    │
 * │   → Never store "total earnings" as a column                       │
 * │   → Never store "course count" as a column                         │
 * │   → Never store "last purchase date" as a column                   │
 * │   → Never store aggregated values as truth                         │
 * │                                                                    │
 * │ WHY THIS MATTERS:                                                  │
 * │                                                                    │
 * │   When you store aggregated values:                                │
 * │     → You cannot audit them                                        │
 * │     → You cannot recalculate them                                  │
 * │     → They become inconsistent over time                           │
 * │     → You lose the ability to verify correctness                   │
 * │                                                                    │
 * │   When you store one fact per row:                                 │
 * │     → Truth is always verifiable                                   │
 * │     → You can always recalculate                                   │
 * │     → History is preserved                                         │
 * │     → Audits are possible                                          │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 4. WHAT IS A "TRUTH TABLE"? — Single source of truth for entities
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ A TRUTH TABLE stores the single source of truth for an entity.     │
 * │                                                                    │
 * │ EXAMPLES:                                                          │
 * │                                                                    │
 * │   students(id, email, name, created_at)                            │
 * │     → One row per student.                                         │
 * │     → This is where student identity is defined.                   │
 * │     → If a student changes name, update ONE row.                   │
 * │                                                                    │
 * │   courses(id, code, title, price, instructor_id)                   │
 * │     → One row per course.                                          │
 * │     → This is where course definition is stored.                   │
 * │     → If course title changes, update ONE row.                     │
 * │                                                                    │
 * │   payments(id, enrollment_id, amount, status, paid_at)             │
 * │     → One row per payment transaction.                             │
 * │     → This is where payment facts are recorded.                    │
 * │     → Each payment is immutable once recorded.                     │
 * │                                                                    │
 * │ PROPERTIES OF A TRUTH TABLE:                                       │
 * │                                                                    │
 * │   ┌─────────────────────────────────────────────────────────┐      │
 * │   │ - Each row represents ONE real-world fact               │      │
 * │   │ - If a value changes, update it in EXACTLY ONE place    │      │
 * │   │ - Always one place, never two                           │      │
 * │   │ - No derived or aggregated values stored                │      │
 * │   │ - All attributes are direct properties of the entity    │      │
 * │   └─────────────────────────────────────────────────────────┘      │
 * │                                                                    │
 * │ THE SIGNAL OF A WRONG MODEL:                                       │
 * │                                                                    │
 * │   If you need to update the same value in multiple rows,           │
 * │   your model is WRONG.                                             │
 * │                                                                    │
 * │   Example:                                                         │
 * │     → Student name appears in 10 rows → wrong                      │
 * │     → Course title appears in 50,000 rows → wrong                  │
 * │     → Commission rate appears in many places → wrong               │
 * │                                                                    │
 * │   CORRECTION:                                                      │
 * │     → Student name belongs in students table (one row)             │
 * │     → Course title belongs in courses table (one row)              │
 * │     → Commission rate belongs in a config or with each sale        │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 5. THE CONVENIENCE TABLE TRAP — Deriving from truth, not replacing it
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ CONVENIENCE TABLES are built to make reads easier or faster.       │
 * │                                                                    │
 * │ EXAMPLES OF CONVENIENCE TABLES:                                    │
 * │                                                                    │
 * │   → Dashboard summary tables (pre-aggregated metrics)              │
 * │   → Reporting tables with pre-joined data                          │
 * │   → Denormalized analytics snapshots                               │
 * │   → Materialized views                                             │
 * │   → Cache tables                                                   │
 * │                                                                    │
 * │ CONVENIENCE TABLES ARE NOT WRONG.                                  │
 * │                                                                    │
 * │ THE TRAP IS:                                                       │
 * │   They must be built FROM truth tables.                            │
 * │   They should NEVER become the source of truth themselves.         │
 * │                                                                    │
 * │ CORRECT PATTERN:                                                   │
 * │                                                                    │
 * │   ┌─────────────────────────────────────────────────────────┐      │
 * │   │                                                         │      │
 * │   │   TRUTH TABLES (Source of Truth)                        │      │
 * │   │   ┌──────────────┐  ┌──────────────┐  ┌──────────────┐  │      │
 * │   │   │  students    │  │  courses     │  │ enrollments  │  │      │
 * │   │   │  (source)    │  │  (source)    │  │  (source)    │  │      │
 * │   │   └──────┬───────┘  └──────┬───────┘  └──────┬───────┘  │      │
 * │   │          │                 │                 │          │      │
 * │   │          └─────────────────┼─────────────────┘          │      │
 * │   │                            │                            │      │
 * │   │                            ▼                            │      │
 * │   │   ┌─────────────────────────────────────────────┐       │      │
 * │   │   │         CONVENIENCE TABLE                   │       │      │
 * │   │   │   (derived, refreshed periodically)         │       │      │
 * │   │   │   dashboard_summary                         │       │      │
 * │   │   │   reporting_snapshot                        │       │      │
 * │   │   │   cache_table                               │       │      │
 * │   │   └─────────────────────────────────────────────┘       │      │
 * │   │                                                         │      │
 * │   └─────────────────────────────────────────────────────────┘      │
 * │                                                                    │
 * │ RULES FOR CONVENIENCE TABLES:                                      │
 * │                                                                    │
 * │   → They are ALWAYS derived from truth tables                      │
 * │   → They can be RECALCULATED from truth tables                     │
 * │   → They are NEVER the only copy of data                           │
 * │   → If they get corrupted, you can rebuild them                    │
 * │   → They exist for PERFORMANCE, not for DATA INTEGRITY             │
 * │                                                                    │
 * │ WHEN CONVENIENCE TABLES BECOME DANGEROUS:                          │
 * │                                                                    │
 * │   → When they become the source of truth                           │
 * │   → When you write to them directly without updating truth         │
 * │   → When you cannot recalculate them from truth                    │
 * │   → When they contain derived values that are also stored          │
 * │     as truth in another place                                      │
 * │                                                                    │
 * │ THE GOLDEN RULE:                                                   │
 * │   Truth tables define reality.                                     │
 * │   Convenience tables derive from reality.                          │
 * │   Never let convenience override truth.                            │
 * └────────────────────────────────────────────────────────────────────┘
 */